# Study: Max Log Price Range Ratio — Full Spatial Statistics

Like `study_max_log_price_range_ratio.ipynb` but instead of one sample per window
(the single most-distant log return), this notebook collects **all 1440 log returns**
inside every complete window and reports their aggregate distribution.

For a window ending at position `i`, the 1440 samples are:
$$\text{log\_return}_{i,j} = \ln\!\left(\frac{\text{vwap}[j]}{\text{vwap}[i]}\right), \quad j = i{-}1439 \,\ldots\, i$$

## RAM strategy

Materialising all values simultaneously would require ~18 GB per asset.
Instead the notebook uses **batch processing + histogram accumulation**:

```
for each batch of BATCH_SIZE windows:
    compute (BATCH_SIZE × 1440) log returns   ← ~20 MB, then discarded
    add counts to a 100-bin histogram          ← 100 integers, kept

peak RAM per asset ≈ 20 MB   (independent of dataset size)
```

Percentiles are recovered by linear interpolation over the accumulated histogram CDF.

In [ ]:
%pip install -q huggingface_hub pyarrow pandas matplotlib requests numpy

In [ ]:
import io
import requests
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi
from IPython.display import display

REPO_ID    = "payamdavaee/candles"
ASSETS     = ["btcusdt", "ethusdt", "trumpusdt", "vineusdt", "adausdt", "xrpusdt", "dogeusdt"]
WINDOW     = 1440
BATCH_SIZE = 2000   # windows per batch; tune down if RAM is tight
BIN_EDGES  = np.linspace(-2.5, 2.5, 101)   # 100 bins; covers ±92 % log-price moves

WIDE_PCTS  = [i / 100 for i in range(0, 101, 10)]   # P0 … P100 step 10
FINE_PCTS  = [i / 100 for i in range(90, 101)]       # P90 … P100 step 1

api       = HfApi()
all_files = sorted(api.list_repo_files(REPO_ID, repo_type="dataset"))
print(f"Total files in dataset: {len(all_files)}")

In [ ]:
def get_parquet_urls(asset: str) -> list[str]:
    prefix = f"{asset}/"
    return [
        f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{f}"
        for f in all_files
        if f.startswith(prefix) and f.endswith(".parquet")
    ]


def load_vwap_array(urls: list[str]) -> np.ndarray:
    """Download parquet files, extract vwap column sorted by ts, return numpy array."""
    tables = []
    for url in urls:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        tables.append(pq.read_table(io.BytesIO(resp.content), columns=["ts", "vwap"]))
    combined = pa.concat_tables(tables).sort_by("ts")
    return combined.column("vwap").to_numpy(zero_copy_only=False).astype(np.float64)


def compute_histogram(
    vwap: np.ndarray,
    window: int = WINDOW,
    batch_size: int = BATCH_SIZE,
    bin_edges: np.ndarray = BIN_EDGES,
) -> tuple[np.ndarray, int, int, int]:
    """
    Memory-efficient histogram of ln(vwap[j] / vwap[i]) for every j in every
    complete window of width `window`.

    Uses numpy sliding_window_view (zero-copy) + batched processing so that
    peak RAM = O(batch_size × window) regardless of total data size.

    Returns
    -------
    counts       : int64 array of length len(bin_edges)-1
    n_windows    : number of complete windows processed
    overflow_low : values below bin_edges[0]  (logged as info)
    overflow_hi  : values above bin_edges[-1] (logged as info)
    """
    N = len(vwap)
    n_windows = N - window + 1

    # Zero-copy view: slide[k] = vwap[k : k+window], right edge = vwap[k+window-1]
    slide = np.lib.stride_tricks.sliding_window_view(vwap, window)

    counts   = np.zeros(len(bin_edges) - 1, dtype=np.int64)
    ov_lo = ov_hi = 0

    for start in range(0, n_windows, batch_size):
        end   = min(start + batch_size, n_windows)
        batch = np.array(slide[start:end])                               # (B, window)
        right = vwap[start + window - 1 : end + window - 1, np.newaxis] # (B, 1)

        lr   = np.log(batch / right)  # (B, window)
        flat = lr.ravel()

        c, _ = np.histogram(flat, bins=bin_edges)
        counts += c
        ov_lo  += int((flat < bin_edges[0]).sum())
        ov_hi  += int((flat > bin_edges[-1]).sum())

    return counts, n_windows, ov_lo, ov_hi


def percentiles_from_hist(
    counts: np.ndarray,
    bin_edges: np.ndarray,
    pct_list: list[float],
) -> dict[float, float]:
    """
    Recover percentile values from histogram counts via linear interpolation
    of the empirical CDF within each bin.
    """
    total = counts.sum()
    cum   = np.concatenate([[0], np.cumsum(counts)])

    results = {}
    for p in pct_list:
        target = p * total
        idx    = int(np.searchsorted(cum[1:], target, side="left"))
        idx    = min(idx, len(counts) - 1)

        lo, hi = cum[idx], cum[idx + 1]
        frac   = (target - lo) / (hi - lo) if hi > lo else 0.5
        results[p] = bin_edges[idx] + frac * (bin_edges[idx + 1] - bin_edges[idx])

    return results


def pct_table(results: dict, pcts: list[float]) -> pd.DataFrame:
    labels = [f"P{int(round(p * 100))}" for p in pcts]
    return pd.DataFrame(
        {"log_return": [results[p] for p in pcts]},
        index=pd.Index(labels, name="percentile"),
    )

In [ ]:
pd.options.display.float_format = "{:.6f}".format

for asset in ASSETS:
    urls = get_parquet_urls(asset)

    print(f"\n{'═' * 62}")
    print(f"  {asset.upper()}   ({len(urls)} monthly file(s))")
    print(f"{'═' * 62}")

    if not urls:
        print("  No Parquet files found — skipping.")
        continue

    vwap = load_vwap_array(urls)
    print(f"  Candles loaded   : {len(vwap):,}")

    counts, n_windows, ov_lo, ov_hi = compute_histogram(vwap)
    total_samples = counts.sum() + ov_lo + ov_hi

    print(f"  Complete windows : {n_windows:,}")
    print(f"  Total log-returns: {total_samples:,}")
    if ov_lo or ov_hi:
        print(f"  Overflow (outside bins): low={ov_lo:,}  high={ov_hi:,}")

    all_pcts = sorted(set(WIDE_PCTS) | set(FINE_PCTS))
    results  = percentiles_from_hist(counts, BIN_EDGES, all_pcts)

    # ── Percentile table: P0 → P100, step 10 ────────────────────────────────
    print("\n  Percentiles  P0 → P100  (step 10)")
    display(pct_table(results, WIDE_PCTS).style.format("{:.6f}"))

    # ── Percentile table: P90 → P100, step 1 ────────────────────────────────
    print("\n  Percentiles  P90 → P100  (step 1)")
    display(pct_table(results, FINE_PCTS).style.format("{:.6f}"))

    # ── Histogram ────────────────────────────────────────────────────────────
    bin_centers = (BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2
    bin_widths  = BIN_EDGES[1:] - BIN_EDGES[:-1]

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(bin_centers, counts, width=bin_widths,
           color="steelblue", edgecolor="white", linewidth=0.3)
    ax.axvline(0, color="crimson", linewidth=1.4, linestyle="--", label="zero")
    ax.set_title(
        f"{asset.upper()} — ln(vwap[j] / vwap[right])  "
        f"all {n_windows:,} windows × {WINDOW} samples  "
        f"[{total_samples:,} points]",
        fontsize=11,
    )
    ax.set_xlabel("log_return")
    ax.set_ylabel("count")
    ax.legend()
    plt.tight_layout()
    plt.show()